## Analysis of EuRepoC Reported Cyber Incidents 
# Goal is to use NLP and machine learning to predict target countries of cyber attacks
Dataset: eurepoc_dyadic_dataset_0_1.csv

**Aim 1**: Using the extracted cyber operation topics identified in build_BERTopic_model.ipynb, are specific types of cyber attacks linked to differences in terms of the intensity of societal impact and political responses? 

**Aim 2**: Can the latent topics emerging from the incident descriptions predict countries targeted by cyber attacks above and beyond the incident types already present in the data set (data theft, disruption, hijacking, physical effects (spatial), physical effects (temporal))?

**Variable codes:** 

weighted_intensity = (data theft + disruption + hijacking + physical effects spatial + physical effect temporal) * effect multiplier [higher for incidents of increasing political importance (critical infrastrucutre, military affected)]
- 1-5 = Low/Moderate Intensity
- 6-10 = High Intensity
- 11-15 = Very High Intensity 

impact_indicator_score = economic impact + political impact + intelligence impact + functional impact 
- 1 = Minor
- 2  Low
- 3 = Medium
- 4 = High
- 5 = Very High



In [1]:
import pandas as pd
import html
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from umap import UMAP
import random
import numpy as np
import spacy
from src.nlp_functions import(remove_geo_entities)
from scipy.stats import kruskal
import scikit_posthocs as sp

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

/Users/heather.iriye/Library/CloudStorage/OneDrive-KarolinskaInstitutet/Documents/GitHub/eurepoc_cyber_security-/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load pre-saved model & data 
topic_model = BERTopic.load(
    "data/topic_model/eurepoc_final_model"
)

topic_assignments = pd.read_csv(
    "data/topic_model/eurepoc_topic_assignments.csv"
)

topic_info = pd.read_csv(
    "data/topic_model/topic_info.csv"
)

dyadic_data = pd.read_csv(
    "data/eurepoc_dataset/dyadic_data_topics.csv"
)

In [7]:
print(dyadic_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 2246 entries, 0 to 2245
Data columns (total 97 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   dyad_id_x                             2246 non-null   int64  
 1   initiator_country_x                   2246 non-null   str    
 2   receiver_country_x                    2246 non-null   str    
 3   incident_id                           2246 non-null   int64  
 4   name_x                                2246 non-null   str    
 5   description_x                         2246 non-null   str    
 6   start_date_x                          2246 non-null   str    
 7   end_date_x                            2246 non-null   str    
 8   source_disclosure_x                   2246 non-null   str    
 9   operation_type_x                      2246 non-null   str    
 10  impact_indicator_score_x              2246 non-null   int64  
 11  impact_indicator_label_x    

In [8]:
# Topic vs intensity
dyadic_data.groupby("topic_group")["weighted_intensity_x"]\
    .mean()\
    .sort_values(ascending=False)

topic_group
Financially Motivated Operations    3.604265
Data Exposure                       3.118557
Information Operations              3.031496
Intrusion Operations                2.517618
Hacktivism                          1.405858
Disruption Operations               1.158416
Name: weighted_intensity_x, dtype: float64

In [ ]:
# Topic vs political responses
dyadic_data.groupby("topic_group")["number_political_responses"]\
    .mean()\
    .sort_values(ascending=False)

In [ ]:
# Most common receiver countries by topic
pd.crosstab(
    dyadic_data["receiver_country"],
    dyadic_data["topic_group"]
)

## Visualizations

In [ ]:
# What cyber conflict themes dominate the dataset?
topic_counts = (
    dyadic_data["topic_group"]
    .value_counts()
    .reset_index()
)

topic_counts.columns = ["topic", "count"]

fig = px.treemap(
    topic_counts,
    path=["topic"],
    values="count",
    title="Distribution of Cyber Conflict Topics"
)

fig.show()

fig.write_html("topic_treemap.html")

In [ ]:
# What cyber conflicts trigger political reactions?
responses = (
    dyadic_data.groupby("topic_group")
    ["number_political_responses"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(10,6))

plt.hlines(
    y=responses.index,
    xmin=0,
    xmax=responses.values
)

plt.plot(
    responses.values,
    responses.index,
    "o"
)

plt.title("Average Political Responses by Topic")
plt.show()

In [ ]:
# How has the nature of cyber conflict changed over time?
dyadic_data["year"] = pd.to_datetime(
    dyadic_data["start_date"],
    errors="coerce"   # invalid dates become NaT
).dt.year

topic_year = pd.crosstab(
    dyadic_data["year"],
    dyadic_data["topic_group"],
    normalize="index"
)

fig, ax = plt.subplots(figsize=(14, 7))

topic_year.plot.area(ax=ax)

ax.set_title("Evolution of Cyber Conflict Topics")
ax.set_ylabel("Share of Incidents")

ax.legend(
    title="Topic",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


## Analysis 

The primary goal of this analysis is to investigate what latent categories of cyber operations emerge form inicident descriptions, and answer how these categories are associated with different initiator countries, target countries, and political outcomes. 

Q1: What operational patterns connect initiator countries, cyber operation types, and receiver countries?
- Visualization: Sankey plot 

Q2: Which operational categories are associated with the most severe incidents?
- Visualization: Boxplot of weighted_intenity
- Statistical Test: Kruskal-Wallis

Q3: What types of cyber operations provoke political responses?
- Visualization: Hline plot

Q4: How have trends in cyber operations evolved over time?
- Visualizaiton: Stacked area chart 

Q5: Which countries are targeted by which types of cyber operations?
- Visualization: Reciever specialization heatmap

Q6: What cyber operation themes dominate the dataset?
- Visualization: Treemap


### Q1: What operational patterns connect initiator countries, cyber operation types, and receiver countries?

In [ ]:
# Filters for sankey visual
top_initiators = (
    dyadic_data["initiator_country"]
    .value_counts()
    .head(10)
    .index
)

top_receivers = (
    dyadic_data["receiver_country"]
    .value_counts()
    .head(10)
    .index
)

sankey_data = dyadic_data[
    dyadic_data["initiator_country"].isin(top_initiators)
    &
    dyadic_data["receiver_country"].isin(top_receivers)
].copy()

In [ ]:
# Shorten topic labels
sankey_data["short_topic"] = (
    sankey_data["topic_group"]
    .str.split(",")
    .str[:3]
    .str.join(", ")
)

In [ ]:
# Create flows
# Initiator -> Topic
flow1 = (
    sankey_data
    .groupby(
        ["initiator_country", "short_topic"]
    )
    .size()
    .reset_index(name="value")
)

# Topic -> Reciever
flow2 = (
    sankey_data
    .groupby(
        ["short_topic", "receiver_country"]
    )
    .size()
    .reset_index(name="value")
)

In [ ]:
# Build node list
nodes = list(
    pd.concat([
        flow1["initiator_country"],
        flow1["short_topic"],
        flow2["receiver_country"]
    ]).unique()
)

node_dict = {
    node: i
    for i, node in enumerate(nodes)
}

In [ ]:
# Build link table
source = []
target = []
value = []

# Initiator -> Topic
for _, row in flow1.iterrows():
    source.append(
        node_dict[row["initiator_country"]]
    )
    target.append(
        node_dict[row["short_topic"]]
    )
    value.append(row["value"])

# Topic -> Receiver
for _, row in flow2.iterrows():
    source.append(
        node_dict[row["short_topic"]]
    )
    target.append(
        node_dict[row["receiver_country"]]
    )
    value.append(row["value"])

In [ ]:
fig = go.Figure(
    go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            label=nodes
        ),
        link=dict(
            source=source,
            target=target,
            value=value
        )
    )
)

fig.update_layout(
    title="Cyber Conflict Flows: Initiator → Topic → Receiver",
    font_size=10
)

fig.write_html("cyber_conflict_sankey.html")

fig.show()


The Sankey plot visualizes the flow of cyber incidents from initiating actors to operational categories and finally to recipient countries. The width of each flow is proportional to the number of incidents observed in the dataset. Cyber incidents are first groupd by initiator country, then aggregated into five broad operational categories derived from BERTopic analysis: **hacktivism, disruption operations, intrusion operations, information operations, data exposure, and financially motivated operations**. The final stage shows the principal recipient coutnries associated with each operational category. 

**Key Findings**
1. **Financially Motivated operations constitute the largest category of incidents**, indicating that ransomware and extortion-related activities are among the most prevalent forms of cyber conflict represented in the dataset.
2. **Attribution remains a major challenge in cyber conflict**, as "Not Attributed" and "Unkknown" represent some of the largest initiating categories in the diagram.
3. **The US is the largest recipient node**, receiving substantial flows across multiple operational categories, particularily financially motivated operations, intrusion operations, data exposure, and disruption operations. 
4. **Intrusion opeerations are strongly linked sith state-linked actors**. China, Russia, and several other state actors are connected to malware operations, cyber espionage, and vulnerabilty exploitation. The pattern is consistent with the use of cyber capabilities for intelligence collection and network penetration. 

# Q2: Which operational categories are associated with the most impactful incidents?

In [ ]:
topic_summary = (
    dyadic_data
    .groupby("topic_group")
    .agg(
        incidents=("incident_id", "count"),
        avg_intensity=("weighted_intensity", "mean"),
        avg_political=("number_political_responses", "mean"),
        avg_legal=("number_legal_responses", "mean")
    )
    .round(2)
)

print(topic_summary)

In [ ]:
plt.figure(figsize=(12,6))

sns.boxplot(
    data=dyadic_data,
    x="topic_group",
    y="weighted_intensity"
)

plt.xticks(rotation=90)
plt.ylabel("Weighted Intensity")
plt.xlabel("Topic Group")
plt.title("Impact Intensity Distribution by Topic")
plt.show()

Kruskal-Wallis test for significant differences between operation types

In [ ]:
# First drop duplicate incident_ids in the dataset
topic_analysis = (
    dyadic_data
    .drop_duplicates(subset=["incident_id"])
    .copy()
)

# Check sample sizes
print(
    dyadic_data.groupby("topic_group")
    .size()
    .sort_values(ascending=False)
)

# Run Kruskal-Wallis test
groups = [
    g["weighted_intensity"].dropna()
    for _, g in topic_analysis.groupby("topic_name")
]

stat, p = kruskal(*groups)

print(f"Kruskal-Wallis H-statistic: {stat:.3f}")
print(f"p-value: {p:.6f}")

alpha = 0.05

if p < alpha:
    print("Significant differences in weighted intensity across topics.")
else:
    print("No significant differences in weighted intensity across topics.")


In [ ]:
# Dunn test
dunn = sp.posthoc_dunn(
    topic_analysis,
    val_col="weighted_intensity",
    group_col="topic_group",
    p_adjust="bonferroni"
)


# Convert Dunn matrix to long format
results = []

for i, row in enumerate(dunn.index):
    for j, col in enumerate(dunn.columns):

        # Keep upper triangle only
        if i < j:

            p = dunn.loc[row, col]

            results.append({
                "Comparison": f"{row} vs {col}",
                "p-value": p,
                "Significant": "Yes" if p < 0.05 else "No"
            })

dunn_table = pd.DataFrame(results)

dunn_table = (
    dunn_table
    .sort_values("Comparison")
    .reset_index(drop=True)
)

dunn_table["p-value"] = (
    dunn_table["p-value"]
    .apply(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")
)

print(dunn_table.to_string(index=False))

## Key Findings
**1. Operational type is associated with severity**
- The Kruskal-Wallis test found significant variation in incident intensity between categories
- The way a cyber operation is conducted appears to be linked ot its observed intensity, with ransomware and extortion representing the most distinct category and hacktivist activities generally representing the least severe

**2. Ransomware & cyber-extortion incidents differed significantly from all other operational categories**
- This suggests that financially motivated attacks exhibit a unique intensity profile compared to other forms of cyber activity

**3. Hacktivism & Disruption Operations occupy the lower-intensity end of the spectrum**
- Both categories differed significantly from most other groups
- Website defacements, symbolic attacks, and service disruptions tend to be less severe than ransomware, espionage, or other intrusion-focused attacks

**4. Data Exposure and Intrusion Operations show similar severity levels**
- Data breaches and instrusion-based operations have comparable intensity characteristics

**5. Information Operations & Intrusion Operations also exhibit similar intensity profiles**
-  Espionage and surveillance oriented incidents appear to operate at a simliar level of severity as malware and exploitation activities

**6. Most operational categories remain distinguishable from each other**
- The majority of pairwise comparisons were significant, indicating meaningful differences in severity across cyber-operation types. 

In [ ]:
pd.crosstab(
    dyadic_data["receiver_country"],
    dyadic_data["topic_name"],
    normalize="index"
)

In [ ]:
topic_analysis.loc[
    topic_analysis["topic_group"].isna(),
    ["topic_name"]
].drop_duplicates()